In [ ]:
!pip install ultralytics opencv-python deep_sort_realtime

In [ ]:
import cv2
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort
from google.colab.patches import cv2_imshow

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
model = YOLO("yolov8n.pt")
tracker = DeepSort(max_age=30)

In [ ]:
video_path = "group of people walkig.mp4"
cap = cv2.VideoCapture(video_path)

unique_ids = set()

In [ ]:
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # YOLO detection
    results = model(frame)[0]

    detections = []

    for box in results.boxes:
        cls_id = int(box.cls[0])
        class_name = model.names[cls_id]

        # only detect people
        if class_name == "person":
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = float(box.conf[0])

            detections.append(([x1, y1, x2, y2], conf, None))

    # DeepSORT tracking
    tracks = tracker.update_tracks(detections, frame=frame)

    for track in tracks:
        if not track.is_confirmed():
            continue

        track_id = track.track_id
        l, t, r, b = map(int, track.to_ltrb())

        # count unique people
        unique_ids.add(track_id)

        # draw bounding box
        cv2.rectangle(frame, (l, t), (r, b), (0, 255, 0), 2)

        # draw ID
        cv2.putText(frame, f"ID {track_id}", (l, t - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    # show total count
    cv2.putText(frame, f"Count: {len(unique_ids)}", (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

    # show frame
    cv2_imshow(frame)